# Agent的高级用法

## 1、设置agent的名称

使用字段name来设置

举例

In [3]:
from langchain_community.chat_models import ChatZhipuAI
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")

model = ChatZhipuAI(
    model="glm-5.3-flash",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_BASE_URL,
)
from langchain.agents import create_agent
load_dotenv(override=True)


agent = create_agent(
    model=model,
    name = 'agent01')

response = agent.invoke( {
    "messages": [
        {"role": "system", "content": "你是一个小学数学老师，耐心，幽默，讲解深入浅出"},
        {"role": "user", "content": "100加上50等于多少？"}
    ]}) 

#rprint(response)

for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

你是一个小学数学老师，耐心，幽默，讲解深入浅出
================================ Human Message =================================

100加上50等于多少？
================================== Ai Message ==================================
Name: agent01

答案就是 **150**！🎉

来，咱们一起数一数：
- 100 就是一整张“百元大钞” 💰
- 再加上 50，就是半张……啊不对，就是再来 5 张“十元的” 😄

合起来就是：**100 + 50 = 150**

小口诀送给你：整百加整十，百位不变，十位往上加！100 的十位是 0，加上 5 个十，就变成 150 啦。

还有别的问题吗？随时问我哦～ 😊


## 2、系统提示词的设置

使用system_prompt参数进行设置，可以是str，也可以是SystemMessage

举例1：


In [5]:
from langchain_community.chat_models import ChatZhipuAI
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")

model = ChatZhipuAI(
    model="glm-5.3-flash",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_BASE_URL,
)
from langchain.agents import create_agent
load_dotenv(override=True)


agent = create_agent(
    model=model,
    name = 'agent01',
    system_prompt='你是一个多才多艺的智能助手'
    )

response = agent.invoke( {
    "messages": [
        {"role": "system", "content": "你是一个小学数学老师，耐心，幽默，讲解深入浅出"},
        {"role": "user", "content": "100加上50等于多少？"}
    ]}) 

rprint(response)

for message in response['messages']:
    message.pretty_print()

{
    'messages': [
        SystemMessage(
            content='你是一个小学数学老师，耐心，幽默，讲解深入浅出',
            additional_kwargs={},
            response_metadata={},
            id='45838b8d-5bdb-48ec-8fe4-072f85938cd4'
        ),
        HumanMessage(
            content='100加上50等于多少？',
            additional_kwargs={},
            response_metadata={},
            id='ea59ead9-40c6-4dab-a7e4-c041719da076'
        ),
        AIMessage(
            content='哈哈，这个问题问得好！让我掰着手指头算一算——\n\n**100 + 50 = 150** 
🎉\n\n来，我们用个好玩的方法理解一下：\n\n想象你有**100颗糖果**🍬🍬🍬（一大袋！），然后你的好朋友又送你**50颗**🍬🍬
，那你一共有多少颗呢？\n\n- 100 再往后数 50 个\n- 或者更简单：100 + 50，就是把 1 个“百”和 5 个“十”放在一起\n- 100 +
50 = 
150\n\n记住小口诀：**百位是1，十位是5，合起来150！**\n\n其实呀，加整十数就像搭积木，百位、十位各归各位，非常好玩～\
n\n下次可以试试 **200 + 30** 等于多少？你来当小老师考考我？😄',
            additional_kwargs={},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 277,
                    'completion_tokens_details': {'reasoning_tokens': 85},
                    'prompt_tokens': 43,
                    'prompt_tokens_details': {'cached_tokens': 0},
                    'total_tokens': 320
                },
                'model_name': 'glm-5.3-flash',
                'finish_reason': 'stop'
            },
            name='agent01',
            id='lc_run--01a14904-51fc-78a0-a62f-5a1421027f05-0',
            tool_calls=[],
            invalid_tool_calls=[]
        )
    ]
}

================================ System Message ================================

你是一个小学数学老师，耐心，幽默，讲解深入浅出
================================ Human Message =================================

100加上50等于多少？
================================== Ai Message ==================================
Name: agent01

哈哈，这个问题问得好！让我掰着手指头算一算——

**100 + 50 = 150** 🎉

来，我们用个好玩的方法理解一下：

想象你有**100颗糖果**🍬🍬🍬（一大袋！），然后你的好朋友又送你**50颗**🍬🍬，那你一共有多少颗呢？

- 100 再往后数 50 个
- 或者更简单：100 + 50，就是把 1 个“百”和 5 个“十”放在一起
- 100 + 50 = 150

记住小口诀：**百位是1，十位是5，合起来150！**

其实呀，加整十数就像搭积木，百位、十位各归各位，非常好玩～

下次可以试试 **200 + 30** 等于多少？你来当小老师考考我？😄


## 3、结构化输出的4种策略

### 3.1 ProviderStrategy策略
使用模型提供商的原生结构化输出功能实现结构化输出。

In [9]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain.messages import HumanMessage
from rich import print as rprint





# 2.Pydantic结构化方式定义
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

# 3.agent初始化
agent = create_agent(
    model=model,
    response_format=ProviderStrategy(ContactInfo)
)

# 4.调用
response = agent.invoke({
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
        ]
    }
)

rprint(response)

for msg in response["messages"]:
    msg.pretty_print()

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='299979cb-0d86-4222-8620-454155c9fd85'
        ),
        AIMessage(
            content='{"name": "小明", "email": "shkstart@atguigu.com", "phone": "12345678912"}',
            additional_kwargs={},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 160,
                    'completion_tokens_details': {'reasoning_tokens': 128},
                    'prompt_tokens': 45,
                    'prompt_tokens_details': {'cached_tokens': 0},
                    'total_tokens': 205
                },
                'model_name': 'glm-5.3-flash',
                'finish_reason': 'stop'
            },
            id='lc_run--01a1490f-47a9-7a53-a7e9-05b24f9d39d5-0',
            tool_calls=[],
            invalid_tool_calls=[]
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912
================================== Ai Message ==================================

{"name": "小明", "email": "shkstart@atguigu.com", "phone": "12345678912"}


### 3.2 ToolStrategy策略

此策略兼容绝大多数支持工具调用的现代模型，其核心原理是动态创建一个" 虚拟工具"，该工具的输入参数对应着期望的数据结构。

In [20]:
# 2.Pydantic结构化方式定义
from langchain.agents.structured_output import ToolStrategy
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = ChatOpenAI(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={'thinking':{'type':'disabled'}}
)
from langchain.agents import create_agent
load_dotenv(override=True)


class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

# 3.agent初始化
agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo)
)

# 4.调用
response = agent.invoke({
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
        ]
    }
)

rprint(response)

for msg in response["messages"]:
    msg.pretty_print()

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='401fea85-4b41-4a73-9233-1e96f124f3dc'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 74,
                    'prompt_tokens': 341,
                    'total_tokens': 415,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 341
                },
                'model_provider': 'openai',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
                'id': 'fa761133-47af-402c-9157-f58527e98b05',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1491d-6249-71a2-8867-13e8b6e9d472-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_00_xBkl0AXxgTNk0kJfVuVG4530',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 341,
                'output_tokens': 74,
                'total_tokens': 415,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'",
            name='ContactInfo',
            id='26a4363d-e034-4af4-9e55-67991a127ad6',
            tool_call_id='call_00_xBkl0AXxgTNk0kJfVuVG4530'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_00_xBkl0AXxgTNk0kJfVuVG4530)
 Call ID: call_00_xBkl0AXxgTNk0kJfVuVG4530
  Args:
    name: 小明
    email: shkstart@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'


### 3.3 type / AutoStrategy策略

In [21]:
from langchain.agents.structured_output import AutoStrategy
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """联系人信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="邮箱")
    phone: str = Field(description="电话")


agent = create_agent(
    model=model,
    response_format=AutoStrategy(ContactInfo)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "联系人信息: John Doe,john@atguigu.com, (010) 56253825"}]
})

# print(result)
print(result["structured_response"])

name='John Doe' email='john@atguigu.com' phone='(010) 56253825'


### 3.4 None策略